In [26]:
# Test cell - Check layer info and test different query approaches
import requests

BASE_URL = "https://gisn.tel-aviv.gov.il/arcgis/rest/services/IView2/MapServer/870"

print("=" * 60)
print("Testing Layer 870 (Scooter Parking) API")
print("=" * 60)

# First, check layer info
print("\n1. Checking layer info...")
info_url = f"{BASE_URL}?f=json"
try:
    info_response = requests.get(info_url, timeout=30)
    info_data = info_response.json()
    print(f"   Status: {info_response.status_code}")
    if "name" in info_data:
        print(f"   Layer name: {info_data.get('name')}")
    if "description" in info_data:
        print(f"   Description: {info_data.get('description', 'N/A')[:100]}")
except Exception as e:
    print(f"   Error getting layer info: {e}")

# Try query with minimal parameters
print("\n2. Testing query with minimal parameters...")
query_url = f"{BASE_URL}/query"
minimal_params = {"f": "json", "returnGeometry": "true", "where": "1=1"}
try:
    response = requests.get(query_url, params=minimal_params, timeout=30)
    data = response.json()
    print(f"   Status: {response.status_code}")
    print(f"   Response keys: {list(data.keys())}")
    
    if "error" in data:
        print(f"   Error: {data['error']}")
    elif "features" in data:
        print(f"   ✓ Success! Found {len(data['features'])} features")
    else:
        print(f"   Unexpected structure: {str(data)[:300]}")
except Exception as e:
    print(f"   Exception: {e}")

# Try query without where clause
print("\n3. Testing query without where clause...")
no_where_params = {"f": "json", "returnGeometry": "true"}
try:
    response = requests.get(query_url, params=no_where_params, timeout=30)
    data = response.json()
    print(f"   Status: {response.status_code}")
    
    if "error" in data:
        print(f"   Error: {data['error']}")
    elif "features" in data:
        print(f"   ✓ Success! Found {len(data['features'])} features")
    else:
        print(f"   Response keys: {list(data.keys())}")
except Exception as e:
    print(f"   Exception: {e}")

print("\n" + "=" * 60)


Testing Layer 870 (Scooter Parking) API

1. Checking layer info...
   Status: 200

2. Testing query with minimal parameters...
   Status: 200
   Response keys: ['error']
   Error: {'code': 500, 'message': None, 'details': []}

3. Testing query without where clause...
   Status: 200
   Error: {'code': 500, 'message': None, 'details': []}



In [27]:
import pandas as pd
import numpy as np
import requests
from pathlib import Path
from math import radians, cos, sin, asin, sqrt

# Get the directory where this notebook is located
# In Jupyter, the current working directory is typically the notebook's directory
NOTEBOOK_DIR = Path.cwd()

# Haversine formula to calculate distance between two points in meters
def haversine_distance(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees)
    Returns distance in meters
    """
    # Convert decimal degrees to radians
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    
    # Radius of earth in meters
    r = 6371000
    
    return c * r

# Calculate centroid of a polygon (simple average of coordinates)
def calculate_polygon_centroid(rings):
    """
    Calculate the centroid of a polygon from its rings
    Returns (lon, lat) tuple
    """
    if not rings or not rings[0]:
        return None
    
    # Use the first ring (outer boundary)
    outer_ring = rings[0]
    
    # Calculate centroid as average of coordinates
    # Note: This is a simple approximation. For more accurate results,
    # we'd need to account for polygon area, but for small parking spots
    # this should be sufficient
    lons = [coord[0] for coord in outer_ring]
    lats = [coord[1] for coord in outer_ring]
    
    centroid_lon = sum(lons) / len(lons)
    centroid_lat = sum(lats) / len(lats)
    
    return (centroid_lon, centroid_lat)


In [28]:
# Fetch bird nests from Bird GBFS API
BIRD_API_URL = "https://mds.bird.co/gbfs/v2/public/tel-aviv-city-center/station_information.json"

print("Fetching bird nests from Bird GBFS API...")
bird_response = requests.get(BIRD_API_URL)
bird_data = bird_response.json()

# Extract station information
bird_nests = []
for station in bird_data.get("data", {}).get("stations", []):
    bird_nests.append({
        "nest_id": station.get("station_id"),
        "name": station.get("name"),
        "address": station.get("address"),
        "latitude": station.get("lat"),
        "longitude": station.get("lon"),
        "is_virtual_station": station.get("is_virtual_station", False),
        "station_area": station.get("station_area")
    })

bird_nests_df = pd.DataFrame(bird_nests)

print(f"Loaded {len(bird_nests_df)} bird nests from Bird API")
print(f"Columns: {bird_nests_df.columns.tolist()}")
print("\nFirst few rows:")
print(bird_nests_df.head())


Fetching bird nests from Bird GBFS API...
Loaded 892 bird nests from Bird API
Columns: ['nest_id', 'name', 'address', 'latitude', 'longitude', 'is_virtual_station', 'station_area']

First few rows:
                                nest_id  \
0  2bcb3825-b52e-4992-91bc-41978f8b8829   
1  54e90aae-a54b-4b15-bea3-d4e20e5f2efa   
2  d1236f08-ea22-4e1b-b8d3-27cb08ef23c8   
3  57c17ae2-3cae-4451-9a6b-13954ee779e9   
4  54af907f-f9f8-4b54-8134-21402c1dfa41   

                                                name  \
0        David Khakhami St 15, Tel Aviv-Yafo, Israel   
1  Menahem Begin Road/HaArbaa, Tel Aviv-Yafo, Israel   
2                  Rival St 5, Tel Aviv-Yafo, Israel   
3      Natan Yellin Mor St 28, Tel Aviv-Yafo, Israel   
4       Mordechai Namir Rd 81, Tel Aviv-Yafo, Israel   

                                             address   latitude  longitude  \
0        David Khakhami St 15, Tel Aviv-Yafo, Israel  32.063230  34.782874   
1  Menahem Begin Road/HaArbaa, Tel Aviv-Yafo, Isra

In [30]:
# Fetch parking spots from TLV City GIS (Layer 870 - Scooter Parking)
URL = "https://gisn.tel-aviv.gov.il/arcgis/rest/services/IView2/MapServer/870/query"

print("Fetching parking spots from TLV City GIS...")

# Try different parameter combinations
# First, try without where clause
params_list = [
    {
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "json",
        "resultRecordCount": 1000  # Limit results per request
    },
    {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": "4326",
        "f": "json",
        "resultRecordCount": 1000
    }
]

city_parking_spots = []
data = None
success = False

for params in params_list:
    try:
        response = requests.get(URL, params=params, timeout=30)
        data = response.json()
        
        # Check for errors
        if "error" in data:
            print(f"Error with params {params}: {data['error']}")
            continue
            
        if "features" not in data:
            print(f"Unexpected response structure. Keys: {data.keys()}")
            continue
            
        # Success - break out of loop
        success = True
        break
        
    except Exception as e:
        print(f"Exception with params {params}: {e}")
        continue

if not success:
    # Try a simpler query
    print("Trying simpler query...")
    simple_params = {
        "f": "json",
        "returnGeometry": "true",
        "outSR": "4326"
    }
    response = requests.get(URL, params=simple_params, timeout=30)
    data = response.json()
    
    if "error" in data:
        print(f"Error from API: {data['error']}")
        print(f"Full response: {data}")
        raise Exception(f"API Error: {data['error']}")
    
    if "features" not in data:
        print(f"Unexpected response structure. Keys: {data.keys()}")
        print(f"Response: {str(data)[:1000]}")
        raise Exception("Response does not contain 'features' key")

# Handle pagination if needed
all_features = []
if "features" in data:
    all_features.extend(data["features"])
    
    # Check if there are more results
    if "exceededTransferLimit" in data and data["exceededTransferLimit"]:
        print("Results exceeded transfer limit, fetching additional pages...")
        offset = len(all_features)
        
        while True:
            pagination_params = {
                "outFields": "*",
                "returnGeometry": "true",
                "outSR": "4326",
                "f": "json",
                "resultOffset": offset,
                "resultRecordCount": 1000
            }
            
            response = requests.get(URL, params=pagination_params, timeout=30)
            page_data = response.json()
            
            if "error" in page_data or "features" not in page_data:
                break
                
            if len(page_data["features"]) == 0:
                break
                
            all_features.extend(page_data["features"])
            offset += len(page_data["features"])
            
            if not page_data.get("exceededTransferLimit", False):
                break

print(f"Processing {len(all_features)} parking spots...")

for feature in all_features:
    geom = feature.get("geometry")
    if not geom or "rings" not in geom:
        continue
        
    rings = geom["rings"]       # polygon boundaries
    
    # Calculate centroid for each parking spot
    centroid = calculate_polygon_centroid(rings)
    
    if centroid:
        city_parking_spots.append({
            "id": feature.get("attributes", {}).get("OBJECTID"),
            "centroid_lon": centroid[0],
            "centroid_lat": centroid[1],
            "attributes": feature.get("attributes", {}),
            "polygon_rings": rings
        })

city_parking_df = pd.DataFrame(city_parking_spots)

print(f"Loaded {len(city_parking_df)} parking spots from City GIS")
if len(city_parking_df) > 0:
    print(f"Columns: {city_parking_df.columns.tolist()}")
    print("\nFirst few rows:")
    print(city_parking_df.head())
else:
    print("Warning: No parking spots loaded!")


Fetching parking spots from TLV City GIS...
Error with params {'outFields': '*', 'returnGeometry': 'true', 'outSR': '4326', 'f': 'json', 'resultRecordCount': 1000}: {'code': 500, 'message': None, 'details': []}
Error with params {'where': '1=1', 'outFields': '*', 'returnGeometry': 'true', 'outSR': '4326', 'f': 'json', 'resultRecordCount': 1000}: {'code': 500, 'message': None, 'details': []}
Trying simpler query...
Error from API: {'code': 500, 'message': None, 'details': []}
Full response: {'error': {'code': 500, 'message': None, 'details': []}}


Exception: API Error: {'code': 500, 'message': None, 'details': []}

In [ ]:
# Match bird nests to city parking spots
# Distance threshold: 5 meters
DISTANCE_THRESHOLD = 5  # meters

# Initialize result columns
bird_nests_df['matched_city_id'] = None
bird_nests_df['distance_to_city_meters'] = None
bird_nests_df['gap_more_than_5m'] = False
bird_nests_df['exists_in_bird_not_city'] = False
bird_nests_df['exists_in_city_not_bird'] = False  # Will be set later

matched_city_ids = set()

print("Matching bird nests to city parking spots...")
for idx, nest in bird_nests_df.iterrows():
    nest_lon = nest['longitude']
    nest_lat = nest['latitude']
    
    min_distance = float('inf')
    closest_city_id = None
    
    # Find the closest city parking spot
    for city_idx, city_spot in city_parking_df.iterrows():
        city_lon = city_spot['centroid_lon']
        city_lat = city_spot['centroid_lat']
        
        distance = haversine_distance(nest_lon, nest_lat, city_lon, city_lat)
        
        if distance < min_distance:
            min_distance = distance
            closest_city_id = city_spot['id']
    
    # Update nest record
    bird_nests_df.at[idx, 'distance_to_city_meters'] = min_distance
    bird_nests_df.at[idx, 'matched_city_id'] = closest_city_id
    
    if min_distance > DISTANCE_THRESHOLD:
        bird_nests_df.at[idx, 'gap_more_than_5m'] = True
        bird_nests_df.at[idx, 'exists_in_bird_not_city'] = True
    else:
        # Mark this city spot as matched
        matched_city_ids.add(closest_city_id)

print(f"Matched {len(matched_city_ids)} city parking spots to bird nests")
print(f"Nests with gap > 5m: {bird_nests_df['gap_more_than_5m'].sum()}")


In [31]:
# Mark city parking spots that don't exist in bird nests
city_parking_df['exists_in_city_not_bird'] = ~city_parking_df['id'].isin(matched_city_ids)

print(f"City parking spots not in bird nests: {city_parking_df['exists_in_city_not_bird'].sum()}")


City parking spots not in bird nests: 33


In [32]:
# Calculate summary statistics
total_bird_nests = len(bird_nests_df)
nests_with_gap_more_5m = bird_nests_df['gap_more_than_5m'].sum()
nests_in_bird_not_city = bird_nests_df['exists_in_bird_not_city'].sum()
nests_matching_city = total_bird_nests - nests_in_bird_not_city
city_spots_not_in_bird = city_parking_df['exists_in_city_not_bird'].sum()
total_city_spots = len(city_parking_df)

print("=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total bird nests: {total_bird_nests}")
print(f"Total city parking spots: {total_city_spots}")
print()
print("Bird Nests Categories:")
print(f"  - Nests matching city location (≤5m): {nests_matching_city}")
print(f"  - Nests with gap > 5m: {nests_with_gap_more_5m}")
print(f"  - Nests in bird but not in city: {nests_in_bird_not_city}")
print()
print("City Parking Spots Categories:")
print(f"  - Spots in city but not in bird: {city_spots_not_in_bird}")
print("=" * 60)


KeyError: 'gap_more_than_5m'

In [ ]:
# Export CSV: Nests in bird but not in city (or gap > 5m)
bird_not_city_df = bird_nests_df[bird_nests_df['exists_in_bird_not_city']].copy()

# Select relevant columns for export
export_columns = ['nest_id', 'name', 'address', 'longitude', 'latitude', 
                  'matched_city_id', 'distance_to_city_meters', 'gap_more_than_5m', 'is_virtual_station']

bird_not_city_export = bird_not_city_df[export_columns].copy()

output_file_1 = NOTEBOOK_DIR / "bird_nests_not_in_city.csv"
bird_not_city_export.to_csv(output_file_1, index=False)

print(f"Exported {len(bird_not_city_export)} nests to: {output_file_1}")
print(f"Columns: {bird_not_city_export.columns.tolist()}")
print("\nFirst few rows:")
print(bird_not_city_export.head())


In [ ]:
# Export CSV: City parking spots not in bird nests
city_not_bird_df = city_parking_df[city_parking_df['exists_in_city_not_bird']].copy()

# Select relevant columns for export
city_export_columns = ['id', 'centroid_lon', 'centroid_lat']
city_not_bird_export = city_not_bird_df[city_export_columns].copy()

# Rename columns for clarity
city_not_bird_export = city_not_bird_export.rename(columns={
    'id': 'city_parking_spot_id',
    'centroid_lon': 'longitude',
    'centroid_lat': 'latitude'
})

output_file_2 = NOTEBOOK_DIR / "city_parking_spots_not_in_bird.csv"
city_not_bird_export.to_csv(output_file_2, index=False)

print(f"Exported {len(city_not_bird_export)} city parking spots to: {output_file_2}")
print(f"Columns: {city_not_bird_export.columns.tolist()}")
print("\nFirst few rows:")
print(city_not_bird_export.head())
